# Data Pre-Processing for Asl-Sign-50

This is the pre-processing notebook for cleaning and feature engineering the media-piped 50 unique sign data from the: Google - Isolated Sign Language Recognition Dataset

## Filtering of Original Data

Implement the following code if starting with the original 250 signs dataset, keep commented if starting from the asl-signs-50.

In [ ]:
# import pandas as pd
# import os
# import shutil

# SELECTED_SIGNS = [
#     "bird", "fish", "duck", "frog", "alligator", "cat", "dog", "cow",
#     "pig", "tiger", "lion", "horse", "wolf", "bee", "owl", "goose",
#     "jump", "dance", "blow", "drink", "drop", "find", "give", "make",
#     "cry", "read", "cut", "hide", "fall", "ride",
#     "yes", "no", "finish", "open", "close", "up", "down", "fast",
#     "quiet", "wait", "now", "later", "every", "same", "any",
#     "pizza", "boat", "airplane", "rain", "snow"
# ]

# TRAIN_CSV    = r".\asl-signs\train.csv" #path to original train.csv file
# LANDMARK_DIR = r".\asl-signs\train_landmark_files" # path to original directory with parquet files (landmark data)
# OUTPUT_DIR   = r".\asl-signs-50" # path to output directory where filtered train.csv and corresponding parquet files will be saved

# train    = pd.read_csv(TRAIN_CSV) # Load original train.csv
# filtered = train[train['sign'].isin(SELECTED_SIGNS)] # Filter to only selected signs

# # Save filtered csv
# os.makedirs(OUTPUT_DIR, exist_ok=True) # Ensure output directory exists
# filtered.to_csv(os.path.join(OUTPUT_DIR, 'train.csv'), index=False) # Save filtered train.csv without index
# print(f"Saved filtered train.csv with {len(filtered)} rows") # Print number of rows in filtered train.csv

# # Copy only the parquet files we need
# copied, missing = 0, 0
# for _, row in filtered.iterrows():  # Iterate over filtered rows to copy corresponding parquet files
#     src = os.path.join(r".\asl-signs", row['path']) # Construct source path for parquet file based on original directory and path in train.csv
#     dst = os.path.join(OUTPUT_DIR, row['path']) # Construct destination path for parquet file in output directory
#     os.makedirs(os.path.dirname(dst), exist_ok=True) # Ensure destination directory exists
#     if os.path.exists(src):
#         shutil.copy2(src, dst)
#         copied += 1
#     else: # If source parquet file is missing, count it
#         missing += 1
#     if copied % 500 == 0:
#         print(f"  Copied {copied} files...")

# print(f"\nDone! Copied {copied} files, {missing} missing")
# print(f"Output saved to: {OUTPUT_DIR}")

## Data Discovery and Exploration

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import os

print("All imports successful")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

All imports successful
NumPy: 1.26.4
Pandas: 2.0.3


In [2]:
# Configuration
DATA_DIR      = './asl-signs-50'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
OUTPUT_DIR    = './Processed_ASL_Data' # output directory to save landmarks
TARGET_FRAMES = 30
MIN_FRAMES    = 5
RANDOM_SEED  = 80
N_HOLDOUT_PARTICIPANTS = 3 # number of participants to hold out for final testing
N_CV_FOLDS    = 5          # number of cross-validation folds

# verify paths exist
print(f"Data dir exists: {os.path.exists(DATA_DIR)}")
print(f"Train CSV exists: {os.path.exists(TRAIN_CSV)}")

# load and preview train.csv
train = pd.read_csv(TRAIN_CSV) # csv that holds the "Table of contents" of the data
print(f"\nTotal samples: {len(train)}")
print(f"Columns: {train.columns.tolist()}")
print(f"\n=== First 5 Rows ===")
print(train.head())
print("\n=== Statistical Summary ===")
print(train.describe())
print("\n=== Column Data Types ===")
print(train.dtypes)

Data dir exists: True
Train CSV exists: True

Total samples: 18907
Columns: ['path', 'participant_id', 'sequence_id', 'sign']

=== First 5 Rows ===
                                            path  participant_id  sequence_id  \
0  train_landmark_files/26734/1000035562.parquet           26734   1000035562   
1  train_landmark_files/28656/1000106739.parquet           28656   1000106739   
2  train_landmark_files/25571/1000210073.parquet           25571   1000210073   
3  train_landmark_files/26734/1000241583.parquet           26734   1000241583   
4  train_landmark_files/27610/1000956928.parquet           27610   1000956928   

   sign  
0  blow  
1  wait  
2  bird  
3  duck  
4   owl  

=== Statistical Summary ===
       participant_id   sequence_id
count    18907.000000  1.890700e+04
mean     33756.297879  2.149644e+09
std      16052.215389  1.231101e+09
min       2044.000000  1.649177e+06
25%      25571.000000  1.094789e+09
50%      32319.000000  2.140024e+09
75%      49445.000000  3

### Data set information
**Train.csv:** is a look up table that maps each video clip to its label, it is a csv of pointers (table of contents for the data).
path -> where is the data file
participant_id -> who signed it  
sequence_id -> unique clip ID
sign -> what word they signed

**Train_Landmark_files/:** This is where the actual data lives with all of the media piped coordinates for each frame of each sign for each sample. <br>
├── 26734/                  //participant ID (one of 21 signers) <br>
│   ├── 1000035562.parquet  //one clip (person signing "a sign name (ex: wolf) here") <br>

**Statistical Summary:**
The count shows that there are no missing values in either numeric column.


In [3]:
# Print all of the different signs
print("\nAll signs and sample counts:") # print the unique signs and how many samples of each sign we have in the training set
print(train['sign'].value_counts().to_string())  # prints the amount each value is in training data
print(f"\nMean Samples for each sign: {train.groupby('sign').size().mean()}")
print(f"\nTotal unique signs: {train['sign'].nunique()}") # print the total number of unique signs in the training set


All signs and sample counts:
sign
duck         405
bird         404
cow          404
cat          400
drink        400
make         398
find         397
owl          396
frog         396
pizza        396
bee          395
tiger        394
up           394
goose        394
airplane     393
lion         392
blow         391
horse        391
cry          390
alligator    390
finish       388
wolf         388
snow         386
yes          386
jump         383
fall         382
rain         381
dog          380
fish         380
same         380
later        377
close        374
no           370
boat         370
now          369
cut          369
pig          368
every        368
hide         363
fast         362
read         362
quiet        358
drop         356
any          355
open         354
ride         347
give         346
wait         346
down         327
dance        312

Mean Samples for each sign: 378.14

Total unique signs: 50


**Initial 50 Signed training sample counts**
There is a decent representation for each sign. The samples for each sign range from 312-405.

In [4]:
# load one parquet file and inspect it
sample_path = os.path.join(DATA_DIR, train['path'][0])
df = pd.read_parquet(sample_path)

print(f"Shape: {df.shape}") # print the shape of the data frame, each row is one landmark for one frame 
print(f"\nFirst few columns: {df.columns.tolist()}")
print(f"\nFirst 3 rows, first 10 columns:")
print(df.iloc[:3, :])
print(f"\nNaN count per frame (first 5 frames):")
print(df.iloc[:5].isna().sum().sum())

Shape: (12489, 7)

First few columns: ['frame', 'row_id', 'type', 'landmark_index', 'x', 'y', 'z']

First 3 rows, first 10 columns:
   frame     row_id  type  landmark_index         x         y         z
0     20  20-face-0  face               0  0.494400  0.380470 -0.030626
1     20  20-face-1  face               1  0.496017  0.350735 -0.057565
2     20  20-face-2  face               2  0.500818  0.359343 -0.030283

NaN count per frame (first 5 frames):
0


**Data Format:**
The data is in long format, each row is one landmark per frame.

We have to restructure the data to define individual samples more clearly for training.

In [5]:
# Looking at a sample before interpolating the data 
print(f"Unique types: {df['type'].unique()}")
print(f"Unique frames: {df['frame'].nunique()}")
print(f"Rows per frame: {len(df) / df['frame'].nunique():.0f}")

# look at just hand rows
hands = df[df['type'].isin(['left_hand', 'right_hand'])]
print(f"\nHand rows: {len(hands)}")
print(f"Hand types: {hands['type'].unique()}")
print(f"Landmark indices: {hands['landmark_index'].unique()}")

Unique types: ['face' 'left_hand' 'pose' 'right_hand']
Unique frames: 23
Rows per frame: 543

Hand rows: 966
Hand types: ['left_hand' 'right_hand']
Landmark indices: [ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20]


**Data Description:**

Landmark types: face, pose, left_hand, right_hand —> we know exactly what to filter; we only want hands <br>
Confirms that there are 543 landmarks per frame <br>
Confirms that there are 21 landmarks per hand, 21*(2 hands)*(3 dimensions x,y,z) = 126 landmark features per frame for hands. <br>

## Data Cleaning 

**Data Leakage:** There is no initial target leakage, and we will normalize and separate data early to prevent train-test contamination

**Missing Values:** There will be two sources of missing data: frames where no hand was detected and actual instances where one of the hand landmarks are zero. We will handle no detections of hands by replacing NaN with 0 and dropping all of the zero frames. This will tell the model that a hand is not present which will be correct for the interpretation of the data, as the zeros carry meaning. If there are specific landmarks within a hand missing, that is very rare so the replacement with zero will still work. 

**Categorial Features:** The label sign classes is a categorical output and we will use ordinal encoding (integers 0-49) made through a LabelEncoder just for lables, it is not adding any meaning or relationships based on the labels. 

**Approach:**
Load only x, y, z columns (this will be faster and simpler)
Reshape assuming exactly 543 rows per frame into (n_frames, 543, 3)
Extract hand landmark indices directly by position since the order is always (as stated by the competition that provided the dataset (https://www.kaggle.com/competitions/asl-signs/overview/evaluation):
    Face: indices 0-467 (468 landmarks)
    Pose: indices 468-500 (33 landmarks)
    Left hand: indices 501-521 (21 landmarks)
    Right hand: indices 522-542 (21 landmarks)

### Data Cleaning Funcion

In [6]:
# The dataset has exactly 543 landmarks per frame:
# face: 0-467, pose: 468-500, left_hand: 501-521, right_hand: 522-542
ROWS_PER_FRAME   = 543
POSE_OFFSET = 468 # pose landmarks start at index 468, so to get the pose landmark indices we add this offset to the landmark index
NOSE_IDX         = POSE_OFFSET + 0   # landmark 468
L_SHOULDER_IDX   = POSE_OFFSET + 11  # landmark 479
R_SHOULDER_IDX   = POSE_OFFSET + 12  # landmark 480
LEFT_HAND_IDX    = slice(501, 522)   # 21 landmarks
RIGHT_HAND_IDX   = slice(522, 543)   # 21 landmarks

# Wrist is landmark index 0 within each hand (index 0 of the 21 landmarks)
# Indices 126-134: nose + left shoulder + right shoulder xyz (9 features)

# ── load_parquet function ─────────────────────────────────────────────────────
# Takes a path to one parquet file (one signing clip)
# Returns a dict {'left', 'right', 'pose'} of cleaned numpy arrays, or None if the clip is unusable
def load_parquet(path):
    
    # load only the x, y, z coordinate columns from the parquet file
    df = pd.read_parquet(path, columns=['x', 'y', 'z'])
    
    # calculate how many frames
    # total rows / 543 landmarks per frame = number of frames
    n_frames = int(len(df) / ROWS_PER_FRAME)
    
    # skip this clip if it has no frames 
    if n_frames == 0:
        return None
    
    # reshape (n_frames*543, 3) array into (n_frames, 543, 3)
    # so we can index by [frame, landmark_index, coordinate]
    # example: data[5, 501, 0] = x coordinate of left wrist in frame 5
    data = df.values.reshape(n_frames, ROWS_PER_FRAME, 3).astype(np.float32) #reshapes the 2d array of x,y,z landmarks into 3D array of frame, landmark, coordinates
    
    # extract landmarks from the full 543-landmark array based off of the slices
    # left hand:  landmarks 501-521 -> shape (n_frames, 21, 3)
    # right hand: landmarks 522-542 -> shape (n_frames, 21, 3)
    left  = data[:, LEFT_HAND_IDX,  :]
    right = data[:, RIGHT_HAND_IDX, :]
    # extract the 3 pose landmarks we care about: nose, left shoulder, right shoulder
    # stack into shape (n_frames, 3, 3) — 3 landmarks, each with x/y/z
    pose = np.stack([
        data[:, NOSE_IDX,       :],
        data[:, L_SHOULDER_IDX, :],
        data[:, R_SHOULDER_IDX, :]
    ], axis=1)
    

    # Replace NaN values with 0
    # NaN occurs when MediaPipe could not detect a hand in a frame
    # I decided to use 0 instead of imputation because: zeros carry meaning that the hand/pose is not present.
    # MediaPipe is all-or-nothing, so either all 21 landmarks or none will be present; the models can learn that zeros = one-handed sign
    left  = np.nan_to_num(left,  nan=0.0)
    right = np.nan_to_num(right, nan=0.0)
    pose  = np.nan_to_num(pose,  nan=0.0)

    
    # identify and remove frames where BOTH hands are completely absent
    left_missing  = np.all(left  == 0, axis=(1, 2))  # True if left hand all zeros
    right_missing = np.all(right == 0, axis=(1, 2))  # True if right hand all zeros
    valid = ~(left_missing & right_missing)          # keep if at least one hand present
    left, right, pose = left[valid], right[valid], pose[valid]
    
    # skip this clip if too few valid frames remain after filtering, removing clips where MediaPipe almost completely failed
    if len(left) < MIN_FRAMES:
        return None
    
    # return the three streams separately — dominance and flattening handled in feature engineering
    # left/right: (T, 21, 3), pose: (T, 3, 3), where T >= MIN_FRAMES and varies per clip
    # T will be standardized to TARGET_FRAMES=30 during interpolation
    return {'left': left, 'right': right, 'pose': pose}


# ── Check on Sample Clip ───────────────────────────────────────────────
# test load_parquet on the first clip in our dataset before running on all 18907
clip = load_parquet(sample_path)
print(f"Left hand shape:  {clip['left'].shape}")   # (T, 21, 3)
print(f"Right hand shape: {clip['right'].shape}")  # (T, 21, 3)
print(f"Pose shape:       {clip['pose'].shape}")   # (T, 3, 3)
print(f"NaN count: {sum(np.isnan(v).sum() for v in clip.values())}")

Left hand shape:  (23, 21, 3)
Right hand shape: (23, 21, 3)
Pose shape:       (23, 3, 3)
NaN count: 0


### Main Data Cleaning Loop

In [7]:
# Main Data Cleaning Loop 
# lists to store cleaned sequences and their labels, "raw" because there is no feature engineering or any changes to the data yet
X_raw, y_raw = [], []

# counter to track when files that are two short get skipped
skipped = 0 

for i, row in tqdm(train.iterrows(), total=len(train), desc="Loading & Cleaning"): # the train.csv holds the path to each parquet for each sign 
    
    # build full path to this clip's parquet file, using the path from the train.csv
    path = os.path.join(DATA_DIR, row['path']) 
    
    # load and clean the parquet file using our load_parquet function
    # returns dict {'left', 'right', 'pose'} or None if clip is unusable
    clip = load_parquet(path)
    
    # skip if clip had too few valid hand frames
    if clip is None:
        skipped += 1
        continue
    
    # store the cleaned clip and its sign label
    X_raw.append(clip)
    y_raw.append(row['sign'])


# ── Cleaning Summary ──────────────────────────────────────────────────────────
print(f"\n=== Data Cleaning Summary ===")
print(f"Total clips:            {len(train)}")
print(f"Successfully loaded:    {len(X_raw)}")
print(f"Total skipped:          {skipped}")
print(f"Retention rate:         {len(X_raw)/len(train)*100:.2f}%")
print(f"\n=== Samples Per Sign After Cleaning ===")
sign_counts = pd.Series(y_raw).value_counts() # counts how many times each sign appears 
print(sign_counts.to_string())
print(f"\nMin samples per sign:  {sign_counts.min()}")
print(f"Max samples per sign:  {sign_counts.max()}")
print(f"Mean samples per sign: {sign_counts.mean():.2f}")
print(f"\nAny sign with 0 samples: {(sign_counts == 0).any()}")

Loading & Cleaning: 100%|██████████| 18907/18907 [00:43<00:00, 436.97it/s]


=== Data Cleaning Summary ===
Total clips:            18907
Successfully loaded:    18848
Total skipped:          59
Retention rate:         99.69%

=== Samples Per Sign After Cleaning ===
duck         405
bird         403
cow          402
cat          400
drink        400
make         398
owl          396
frog         396
find         395
pizza        395
tiger        394
up           394
goose        394
bee          394
lion         392
airplane     391
alligator    390
horse        390
blow         390
cry          388
wolf         386
finish       386
yes          385
snow         384
fall         381
fish         380
rain         380
jump         380
dog          379
later        376
same         376
close        373
no           368
boat         368
now          368
cut          368
pig          368
every        366
hide         363
fast         361
read         359
open         354
any          354
quiet        354
drop         353
wait         345
ride         345
give       

After cleaning the data we maintain 99.7 percent of the data, which is great.
The mean samples per sign decreases from 378.14 to 376.96, which is a negligable decrease.

In [8]:
# Convert to numpy arrays 
# X_raw is a list of dicts {'left': (T, 21, 3), 'right': (T, 21, 3), 'pose': (T, 3, 3)}
# we can't stack them yet since T isn't the same for every clip; we'll convert y_raw to numpy now and handle X after feature engineering and normalization

y_raw = np.array(y_raw)

print(f"=== Raw Data Summary ===")
print(f"Total clips loaded: {len(X_raw)}")
print(f"Labels shape: {y_raw.shape}")
print(f"\nSequence length distribution:")
lengths = np.array([len(s['left']) for s in X_raw])
print(f"  Min frames:    {min(lengths)}")
print(f"  Max frames:    {max(lengths)}")
print(f"  Mean frames:   {np.mean(lengths):.1f}")
print(f"  Median frames: {np.median(lengths):.1f}")
print(f"Left hand shape per clip:  {X_raw[0]['left'].shape}")   # (T, 21, 3)
print(f"Right hand shape per clip: {X_raw[0]['right'].shape}")  # (T, 21, 3)
print(f"Pose shape per clip:       {X_raw[0]['pose'].shape}")   # (T, 3, 3)

=== Raw Data Summary ===
Total clips loaded: 18848
Labels shape: (18848,)

Sequence length distribution:
  Min frames:    5
  Max frames:    422
  Mean frames:   37.4
  Median frames: 22.0
Left hand shape per clip:  (23, 21, 3)
Right hand shape per clip: (23, 21, 3)
Pose shape per clip:       (23, 3, 3)


In [9]:
# Distribution of Frames per Sample
lengths = np.array([len(s['left']) for s in X_raw])

print("=== Frame count distribution ===")
thresholds = [5, 6, 8, 10, 12, 15, 20, 30]
for t in thresholds:
    below = np.sum(lengths < t)
    above = np.sum(lengths >= t)
    print(f"  < {t:3d} frames: {below:6d} ({below/len(lengths)*100:6.2f}%)  |  >= {t:3d} frames: {above:6d} ({above/len(lengths)*100:6.2f}%)")

=== Frame count distribution ===
  <   5 frames:      0 (  0.00%)  |  >=   5 frames:  18848 (100.00%)
  <   6 frames:     15 (  0.08%)  |  >=   6 frames:  18833 ( 99.92%)
  <   8 frames:   3019 ( 16.02%)  |  >=   8 frames:  15829 ( 83.98%)
  <  10 frames:   3845 ( 20.40%)  |  >=  10 frames:  15003 ( 79.60%)
  <  12 frames:   4806 ( 25.50%)  |  >=  12 frames:  14042 ( 74.50%)
  <  15 frames:   6274 ( 33.29%)  |  >=  15 frames:  12574 ( 66.71%)
  <  20 frames:   8569 ( 45.46%)  |  >=  20 frames:  10279 ( 54.54%)
  <  30 frames:  11835 ( 62.79%)  |  >=  30 frames:   7013 ( 37.21%)


**Frame Distrubution:**
Based on the distribution of the frames per sign, and the stated 25fps for the clips. I currently think it is best to keep the target frames at 30, as compression would lose information and would most likely cause more harm than stretching the sings with less frames.

The tradeoff was considered and might be revisited based on the results of all the training models.

## Feature Engineering


**Features for this project are the 137 positional features per frame:**
- 126 hand features: x, y, z for each of 21 landmarks per hand (dominant + non-dominant)
- 9 pose features: x, y, z for nose, left shoulder, and right shoulder
- 2 presence mask flags: whether the dominant and non-dominant hand are detected in each frame

**Planned Feature Eningeering:**

Interpolation: will standardize all the clip lengths stretching frames by duplicating and connecting landmark values or compressing the frames to fit within the target frames. 

Dominant-hand canonicalization: rather than treating left and right hand as fixed separate for each sign, we will determine which hand is more present (dominant) across the clip and label it as the signer's dominant hand. This makes the representationc such that the same sign performed by a left-handed or right-handed signer maps to the same feature layout. This will add two features per frame dictating if the dominant hand of the clip (hand present in most frames of the clip) is present in the current frame. We will canonicalize the hands appearing to help the model better learn which hand the signer is using more dominantly. 


Normalization: There will be scale and position normalization of the hands. Position normalization centers all hand landmarks relative to the midpoint between the two wrists, removing position varaince from different recording angles and distances. Scale normalization then will divide by the mean distance of all detected landmarks from that center point, making the representation invariant to hand size and distance from the camera. 
Pose landmarks (nose, shoulders) are normalized separately, centered on the shoulder midpoint and scaled by shoulder width.

Velocity features: computing how much each landmark moves between frames captures motion patterns, which is important since many signs look similar in static frames but have different movements. This doubles the feature size to 274 per frame.

For Random Forest specifically: since RF can't learn temporal patterns, we need to engineer temporal features manually: mean, std, min, max, displacement of each coordinate across all frames. This is what the professor's comment #5 was asking about. This will be done in the random forest training notebook.



### Dominant-Hand Canonicalization

**Canonicalize_hands**: takes the cleaned dict {'left', 'right', 'pose'} and returns (dom, non, pose) arrays where dom is whichever hand appeared in more frames across the clip. This decision is made once per clip.

In [10]:
# Takes the cleaned dict from load_parquet and determines which hand is dominant
# based on presence across the clip. Returns (dom, non, pose) arrays.
def canonicalize_hands(clip):
    left  = clip['left']   # (T, 21, 3)
    right = clip['right']  # (T, 21, 3)
    pose  = clip['pose']   # (T, 3, 3)

    # count how many frames each hand is present (any non-zero landmark) across the clip
    left_present  = np.any(left  != 0, axis=(1, 2))  # (T,) bool array
    right_present = np.any(right != 0, axis=(1, 2))  # (T,) bool array

    # whichever hand appears in more frames becomes dominant for the entire clip
    if right_present.sum() >= left_present.sum():
        dom, non = right, left
    else:
        dom, non = left, right

    return dom, non, pose

### Feature Flattening

**to_flat_features**: takes (dom, non, pose) and builds the final (T, 137) feature vector per frame:
- Indices 0-62:   dominant hand xyz (21 landmarks × 3)
- Indices 63-125: non-dominant hand xyz (21 landmarks × 3)
- Indices 126-128: nose xyz
- Indices 129-131: left shoulder xyz
- Indices 132-134: right shoulder xyz
- Indices 135-136: presence mask — 1.0 if dominant/non-dominant hand detected in that frame, 0.0 if not

In [11]:
# Takes (dom, non, pose) arrays and builds the final (T, 137) feature vector.
# This is where the feature layout is defined.
N_POS_FEATURES = 137  # 126 hand + 9 pose + 2 presence mask flags

def to_flat_features(dom, non, pose):
    # dom:  (T, 21, 3) dominant hand landmarks
    # non:  (T, 21, 3) non-dominant hand landmarks
    # pose: (T, 3, 3)  nose, left shoulder, right shoulder
    T = len(dom)
    out = np.zeros((T, N_POS_FEATURES), dtype=np.float32)


    # Wrist is landmark index 0 within each hand
    # dominant wrist:     feature index 0 (x), 21 (y), 42 (z)
    # non-dominant wrist: feature index 63 (x), 84 (y), 105 (z)
    
    # dominant hand xyz
    out[:, 0:21]    = dom[:, :, 0]  # dominant hand x coords
    out[:, 21:42]   = dom[:, :, 1]  # dominant hand y coords
    out[:, 42:63]   = dom[:, :, 2]  # dominant hand z coords

    # non-dominant hand xyz
    out[:, 63:84]   = non[:, :, 0]  # non-dominant hand x coords
    out[:, 84:105]  = non[:, :, 1]  # non-dominant hand y coords
    out[:, 105:126] = non[:, :, 2]  # non-dominant hand z coords

    # pose landmarks xyz
    out[:, 126:129] = pose[:, 0, :]  # nose xyz
    out[:, 129:132] = pose[:, 1, :]  # left shoulder xyz
    out[:, 132:135] = pose[:, 2, :]  # right shoulder xyz

    # presence mask per frame
    # uses any non-zero landmark as the detection signal to tell if the hand is present in the frame
    out[:, 135] = np.any(dom != 0, axis=(1, 2)).astype(np.float32)  # dominant hand present
    out[:, 136] = np.any(non != 0, axis=(1, 2)).astype(np.float32)  # non-dominant hand present

    return out  # (T, 137)


# ── Test on sample clip ───────────────────────────────────────────────────────
dom, non, pose = canonicalize_hands(clip)
seq = to_flat_features(dom, non, pose)
print(f"After canonicalize_hands:")
print(f"  Dominant hand shape:     {dom.shape}")   # (T, 21, 3)
print(f"  Non-dominant hand shape: {non.shape}")   # (T, 21, 3)
print(f"  Pose shape:              {pose.shape}")  # (T, 3, 3)
print(f"\nAfter to_flat_features:")
print(f"  Sequence shape: {seq.shape}")            # (T, 137)
print(f"  NaN count:      {np.isnan(seq).sum()}")
print(f"  Presence mask (first 5 frames):")
print(f"    dom present: {seq[:5, 135]}")
print(f"    non present: {seq[:5, 136]}")

After canonicalize_hands:
  Dominant hand shape:     (27, 21, 3)
  Non-dominant hand shape: (27, 21, 3)
  Pose shape:              (27, 3, 3)

After to_flat_features:
  Sequence shape: (27, 137)
  NaN count:      0
  Presence mask (first 5 frames):
    dom present: [1. 1. 1. 1. 1.]
    non present: [1. 1. 1. 1. 1.]


### Interpolation

We need to stretch and compress the different samples with n amount of frames to adheer to the target frames that we have set.

We are using linear interpolation here as it is faster and and will not overshoot/create landmarks that did not exist before. We are keeping the target frames higher, as described earlier, to prevent loss of meaningful features from compression of the longer sample clips/signs.

Non-linear interpolation, especially for compressing the longer clips, was considered. It would maintain the variance of the data better if we remove the clips similar to those before and after it. Ultimiatly this would create variation of inconsistent timesteps in the data which could confuse some of the modles, like LSTM. Addiitionally, the inference predictions would need to be more complicated and interpolation as a whole would be more complicated.

In [12]:
# Interpolation 
# Resamples every sequence to exactly TARGET_FRAMES=30 frames
# using linear interpolation along the time axis, adding or compressing the amount of frames per sample

# Short clips get stretched to 30 — frames get duplicated/interpolated
# Long clips get compressed to 30 — frames get subsampled

# interpolate function for each sample clip
def interpolate(seq, target=TARGET_FRAMES):
    T = len(seq)
    
    # if already correct length, return as is
    if T == target:
        return seq
    
    # create evenly spaced points from 0 to 1 for original and target lengths
    # think of these as timestamps: original clip has T timestamps, we want 30
    x_old = np.linspace(0, 1, T)       # T points between 0 and 1
    x_new = np.linspace(0, 1, target)  # 30 points between 0 and 1

    # checks to make sure x_old is increasing, as it is a requirment of the np.interp, should always be correct anyway
    if not np.all(np.diff(x_old) > 0): return False
    
    # create output array of shape (target, 137) for the 137 features (126 hand + 9 pose + 2 hand presence indicators)
    output = np.zeros((target, seq.shape[1]), dtype=np.float32)
    
    # interpolate each feature independently across the clip using np.interp
    for i in range(seq.shape[1]):
        output[:, i] = np.interp(x_new, x_old, seq[:, i])
    
    return output # result shape: (target, 137)
    

# ── Test on sample clip ───────────────────────────────────────────────────────
seq_interp = interpolate(seq)  # seq is the (T, 137) output from to_flat_features above
print(f"Before interpolation: {seq.shape}")
print(f"After interpolation:  {seq_interp.shape}")  # (30, 137)
print(f"NaN count after interpolation: {np.isnan(seq_interp).sum()}")

Before interpolation: (27, 137)
After interpolation:  (30, 137)
NaN count after interpolation: 0


Used the numpy np.interp function to interpolate the frames for each sign (https://numpy.org/doc/stable/reference/generated/numpy.interp.html). This function works to fill in missing values between known data points using straight lines. The target data points are redfined using the "x_new = np.linspace(0,1,target)".

We are running canonicalization, flattening, interpolation, normalization, and velocity features over the data in one loop in the Feature Engineering section below.


### Normalization

We need to do normalization based on the description of the data set. 

Data inconsistencies:
- Different Recording Positions: The signers had the recordings of the signing actions taken from different heights and angles. Wrist normalization for position is important for the same reason CNNs use pooling and convolution to reduce dependancy on position.
- Different Hand Sizes: The different hand sizes means the wrist at different positions means different things based on the signer. Wrist normalization should center this regardless of hand size.
- Real-time inference: During inference on real time data the wrist and angles will be at different positions than in the training data samples. Without normilization, the model will get confused, having never seen the hand(s) on the screen in the given shape and position.
- Left vs. Right Hand Signers: Some signers use different hands as their main hand sign. Normalization and later Dominant-hand canonicalization should help midigate the difference for this.
- Different Recording Distances for Pose: The absolute position of the nose and shoulders varies drastically across signers based on how far they are from the camera and their position in the frame. 


Normalization Approaches
- Original Idea for Normalization:  will be done by subtracting the wrist positon from each landmark coordinate (for each axis) and storing the result as a replacement of the original position of that landmark. This will be done for both hands, given that the hand appears in the frame. The problem with this is that it sets both wrists to zero and loses the spatial relationship between the two hands that is important for signs using two hands.
- Including both normalized and non-normalized data: This would double the feature size and create a lot of dependent redundancy, which is not the best option for this application.
- feature engineering using the original idea and adding a feature for hand distance separation: this is a better option, but still loses a lot of the spatial relationships between the hand landmarks.
- Normalization of the hands relative to a single reference:  this is the normalization choice we chose as it keeps the spatial relationship while normalizing the wrist data. For this method we will make the reference point the middle point between the two wrists, normal normalization happens with just one hand.
- Scale Normalization: After centering around the reference point, landmarks from different signers still are differenet in scale due to different hand sizes and distance from the camera while recording. We normalize by the mean distance of all detected landmarks from the new origin to make hand size invariant. This is important to normalize as different sized/distance hands can look like different signs as the landmarks are changing by different ratios/distances based onthe scales of the hands.
- Pose normalization: Pose landmarks (nose, shoulders) are normalized separately from the hands. The shoulder midpoint is used as the origin for position normalization and shoulder width is used as the scale factor for scale normalization. This would make the pose features invariant to the signer's position and distance from the camera.

We will use the afformentioned Normalization around a single reference point, which is the mid point between the two wrist coordinates. This normalization methods will make the landmarks position invariant, which is good for inference on new data for our models. Additionally, we will use the scale normalization to allow the landmarks/signs to be scale invariant, which will also help with accuracy on unseen data. We will also ensure this normalization is applied to only detected hands, to maintain the meaning of zeros representing a missing hand. Pose landmarks are additionally normalized relative to the shoulder midpoint and scaled by shoulder width. Frames where both shoulders are missing have their pose features zeroed out entirely.


In [13]:
# Wrist Normalization 
# Normalizes hand landmarks relative to the midpoint between the two wrists, and pose landmarks relative to the shoulder midpoint scaled by shoulder width.
# This preserves the spatial relationship between hands/poses while achieving position and scale invariance.

# Edge cases (hands):
# - Only dominant hand detected:  midpoint = dominant wrist
# - Only non-dominant hand detected:  midpoint = non-dominant wrist
# - Neither hand detected:  skip hand normalization entirely
# - Both hands detected: midpoint = average of both wrists
# Edge cases (pose):
# - Both shoulders detected: midpoint = shoulder midpoint, scale = shoulder width
# - Either shoulder missing: zero out pose features entirely for that frame

# normalize each sample clip individually with this normalize function
def normalize(seq):
    seq = seq.copy()  # don't modify the original array
    
    
    for frame in range(len(seq)): # for each frame in the clip 
        #Hand Normalization
        # get dominant wrist position (index 0=x, 21=y, 42=z)
        dx, dy, dz = seq[frame, 0], seq[frame, 21], seq[frame, 42]

        # get non-dominant wrist position (index 63=x, 84=y, 105=z)
        nx, ny, nz = seq[frame, 63], seq[frame, 84], seq[frame, 105]
        
        # determine which hands are detected in this frame
        dom_detected = not (dx == 0 and dy == 0 and dz == 0)
        non_detected = not (nx == 0 and ny == 0 and nz == 0)
        
        # skip this frame entirely if neither hand is detected
        if not dom_detected and not non_detected:
            seq[frame, 126:135] = 0  # still zero out pose
            continue
        
        # compute reference point based on which hands are present
        if dom_detected and non_detected: # if both hands are detected
            # both hands detected — use midpoint between wrists
            ref_x = (dx + nx) / 2
            ref_y = (dy + ny) / 2
            ref_z = (dz + nz) / 2

        elif dom_detected:
            # only dominant hand — use dominant wrist as reference
            ref_x, ref_y, ref_z = dx, dy, dz
        else:
            # only non-dominant hand — use non-dominant wrist as reference
            ref_x, ref_y, ref_z = nx, ny, nz
                
        # subtract reference point from ALL hand landmarks (for hands that are detected), preserving their spatial relationship
        if dom_detected:
            seq[frame, 0:21]  -= ref_x   # dominant hand x coords
            seq[frame, 21:42] -= ref_y   # dominant hand y coords
            seq[frame, 42:63] -= ref_z   # dominant hand z coords

        if non_detected:
            seq[frame, 63:84]   -= ref_x  # non-dominant hand x coords
            seq[frame, 84:105]  -= ref_y  # non-dominant hand y coords
            seq[frame, 105:126] -= ref_z  # non-dominant hand z coords

        # Scale Normalization 
        
        # gather all x, y, z coords from both hands for unified scale computation
        xs = np.concatenate([seq[frame, 0:21],   seq[frame, 63:84]]) 
        ys = np.concatenate([seq[frame, 21:42],  seq[frame, 84:105]])
        zs = np.concatenate([seq[frame, 42:63],  seq[frame, 105:126]])
        
        # only include landmarks from detected hands (exclude zeros from undetected hand)
        mask = ~((xs == 0) & (ys == 0) & (zs == 0)) # 
        
        # skip scale normalization if no valid landmarks remain (should not happen, just a failsafe)
        if mask.sum() == 0:
            continue
        
        # compute mean distance of detected landmarks from the origin, this represents the how far landmarks typically sit from the wrist center of the hand in this frame
        # a larger or closer hand will have a larger scale than a smaller or farther hand.
        scale = np.sqrt(xs[mask]**2 + ys[mask]**2 + zs[mask]**2).mean()
        
        # skip if the scale is close zero to avoid division by zero and resulting large numbers when dividing coordinates by the scale
        if scale < 1e-6:
            seq[frame, 126:135] = 0  # still zero out pose
            continue
        
        # divide only hand coordinates by scale — pose block handled separately below
        # undetected hand features stay zero since 0 / scale = 0
        seq[frame, 0:126] /= scale


        # Pose Normalization 
        # normalize nose and shoulders relative to shoulder midpoint, scaled by shoulder width
        # seq[frame, 126:129] = nose xyz
        # seq[frame, 129:132] = left shoulder xyz
        # seq[frame, 132:135] = right shoulder xyz
        # Read and check if shoulder landmarks are present to determine if pose normalization can be applied, since MediaPipe pose is less reliable than hands and often fails to detect shoulders
        lsx, lsy, lsz = seq[frame, 129], seq[frame, 130], seq[frame, 131]
        rsx, rsy, rsz = seq[frame, 132], seq[frame, 133], seq[frame, 134]
        ls_ok = not (lsx == 0 and lsy == 0 and lsz == 0)
        rs_ok = not (rsx == 0 and rsy == 0 and rsz == 0)

        if ls_ok and rs_ok:
            # shoulder midpoint as origin
            cx = (lsx + rsx) / 2
            cy = (lsy + rsy) / 2
            cz = (lsz + rsz) / 2

            # shoulder width as scale factor (xy only, z less reliable from MediaPipe)
            sh_w = np.sqrt((lsx - rsx)**2 + (lsy - rsy)**2) # gets shoulder width for x/y plane
            if sh_w >= 1e-6: # if width is not zero
                for base in (126, 129, 132): # for each of the three pose landmarks (nose, left shoulder, right shoulder)
                    seq[frame, base]     -= cx # shift x coords by shoulder midpoint x
                    seq[frame, base + 1] -= cy # shift y coords by shoulder midpoint y
                    seq[frame, base + 2] -= cz # shift z coords by shoulder midpoint z
                seq[frame, 126:135] /= sh_w # scale by shoulder width to achieve size invariance, so that people with different shoulder widths can perform the same sign and have the same normalized pose features
            else:
                seq[frame, 126:135] = 0 # if shoulders width is zero, zero out pose features
        else:
            # either shoulder missing -> zero out pose features entirely
            seq[frame, 126:135] = 0

    return seq

# ── Test on sample clip ───────────────────────────────────────────────────────
seq_norm = normalize(seq_interp)  # seq_interp is the (30, 137) output from interpolate above
print(f"Shape after normalization: {seq_norm.shape}")  # (30, 137)
print(f"NaN count: {np.isnan(seq_norm).sum()}")
print(f"Value range: [{seq_norm.min():.3f}, {seq_norm.max():.3f}]")

Shape after normalization: (30, 137)
NaN count: 0
Value range: [-3.114, 2.058]


### Velocity Features

Computing frame-to-frame differences for each of the 137 positional features captures motion patterns across the clip. A lot of the signs look similar in static frames but differ in how the hands move, so velocity gives the model motion information.

Each frame's velocity is the difference between that frame and the previous one. Frame 0 has zero velocity by convention since there is no previous frame.

This doubles the feature size from 137 -> 274 per frame.

In [14]:
N_FEATURES = 274  # 137 positional + 137 velocity features

def add_velocity(seq):
    # seq: (T, 137) normalized positional features
    vel = np.zeros_like(seq)          # (T, 137) all zeros
    vel[1:] = seq[1:] - seq[:-1]      # frame-to-frame differences, frame 0 stays zero
    return np.concatenate([seq, vel], axis=1)  # (T, 274)

# ── Test on sample clip ───────────────────────────────────────────────────────
seq_vel = add_velocity(seq_norm)
print(f"Before add_velocity: {seq_norm.shape}")   # (30, 137)
print(f"After add_velocity:  {seq_vel.shape}")    # (30, 274)
print(f"NaN count: {np.isnan(seq_vel).sum()}")
print(f"Velocity frame 0 (should be all zeros): {np.all(seq_vel[0, 137:] == 0)}")
print(f"Velocity frame 1 sample: {seq_vel[1, 137:140]}")  # first 3 velocity features

Before add_velocity: (30, 137)
After add_velocity:  (30, 274)
NaN count: 0
Velocity frame 0 (should be all zeros): True
Velocity frame 1 sample: [-0.00808285  0.00405586 -0.01265509]


### Apply All The Feature Engineering 

Applies the Dominant-hand canonicalization, Feature flattening, interpolation, position normalization, scale normalization, velocity features to all the data as described above. 

The output X_all has shape (N, 30, 274) where N is the number of clips.

In [15]:
# Apply Feature Engineering to All Clips
# Also tracks participant_id alongside each clip for group-aware splitting later
X_all, y_all, groups_all = [], [], []

for i, clip in enumerate(tqdm(X_raw, desc="Feature Engineering")):
    # canonicalize hands — determine dominant/non-dominant for this clip
    dom, non, pose = canonicalize_hands(clip)
    
    # flatten to (T, 137) feature vector
    seq = to_flat_features(dom, non, pose)
    
    # resample to fixed 30 frames
    seq = interpolate(seq)
    
    # center, scale normalize hands and pose
    seq = normalize(seq)
    
    # append velocity features — doubles feature size to 274
    seq = add_velocity(seq)
    
    X_all.append(seq)
    y_all.append(y_raw[i])                        # string sign label for this clip
    groups_all.append(train.iloc[i]['participant_id'])  # signer ID for group-aware split


# convert to numpy arrays
X_all = np.array(X_all, dtype=np.float32)  # (N, 30, 274)
y_all = np.array(y_all)                    # (N,) string labels
groups_all = np.array(groups_all)          # (N,) participant IDs

print(f"\nFinal dataset shape: {X_all.shape}")
print(f"Labels shape:        {y_all.shape}")
print(f"Groups shape:        {groups_all.shape}")
print(f"NaN count:           {np.isnan(X_all).sum()}")
print(f"Unique participants: {len(np.unique(groups_all))}")

Feature Engineering: 100%|██████████| 18848/18848 [00:26<00:00, 721.79it/s]



Final dataset shape: (18848, 30, 274)
Labels shape:        (18848,)
Groups shape:        (18848,)
NaN count:           0
Unique participants: 21


### Label Encoding 

Converting each string sign labels to integers to represent each class/sign as an integer.

Keeping the le so it can be reversed at inference time,

In [16]:

import joblib

# save the encoder
# joblib.dump(le, './data/landmarks/label_encoder.pkl')

le = LabelEncoder()
y_enc = le.fit_transform(y_all)  # fit on all labels, transform to integers

print(f"Classes: {len(le.classes_)}")
print(f"Class names: {list(le.classes_)}")

# load it later in training notebook
# le = joblib.load('./data/landmarks/label_encoder.pkl')
# then use it

# le.inverse_transform([3])  

Classes: 50
Class names: ['airplane', 'alligator', 'any', 'bee', 'bird', 'blow', 'boat', 'cat', 'close', 'cow', 'cry', 'cut', 'dance', 'dog', 'down', 'drink', 'drop', 'duck', 'every', 'fall', 'fast', 'find', 'finish', 'fish', 'frog', 'give', 'goose', 'hide', 'horse', 'jump', 'later', 'lion', 'make', 'no', 'now', 'open', 'owl', 'pig', 'pizza', 'quiet', 'rain', 'read', 'ride', 'same', 'snow', 'tiger', 'up', 'wait', 'wolf', 'yes']


## Splitting the Data

Using a hold out to seperate N signers so that when inference is made from a never before seen signer it can have accurate predictions on it. This will change the split from 70/15/15 to a split based on the number of participants and the hold out participants. The split will still follow close to the origional split of 70/15/15, but will be a bit rougher.

We will also use cross validation for better training results. 

### Group-Aware Holdout Test Split 
With a regular split, clips from the same signer would appear in both train and test sets. This would cause the model to learn/memorize signer patterns (hand size, style) rather than the actual sign, making test accuracy appear higher than the performance on unseen signers.

GroupShuffleSplit solves this by splitting the data at the signer level, entire participants are either in dev (train and validation) or test; they are never in both.

With 21 total participants and N_HOLDOUT_PARTICIPANTS=3:
- Test set: 3 participants (~14% of data) 
- Dev(Train+validation) set:  18 participants (~86% of data)

In [17]:
# get all unique participant IDs
unique_participants = np.unique(groups_all)
print(f"Total participants: {len(unique_participants)}")
print(f"Holding out {N_HOLDOUT_PARTICIPANTS} participants for test set")

# compute what fraction of participants to hold out
holdout_frac = N_HOLDOUT_PARTICIPANTS / len(unique_participants)

# GroupShuffleSplit splits by participant not by individual sample
# n_splits=1: we only need one split
# test_size:  fraction of participants to hold out
gss = GroupShuffleSplit(n_splits=1, test_size=holdout_frac, random_state=RANDOM_SEED)

# generate the split indices
dev_idx, test_idx = next(gss.split(X_all, y_enc, groups=groups_all)) # next i

# slice arrays using the split indices
X_dev,  y_dev,  groups_dev  = X_all[dev_idx],  y_enc[dev_idx],  groups_all[dev_idx]
X_test, y_test, groups_test = X_all[test_idx], y_enc[test_idx], groups_all[test_idx]

print(f"\nDev set:  {X_dev.shape} -> {len(np.unique(groups_dev))} participants")
print(f"Test set: {X_test.shape} -> {len(np.unique(groups_test))} participants")
print(f"Test participants: {np.unique(groups_test)}")

# verify no participant appears in both dev and test 
print(f"\nNo participant appears in both sets: {len(set(groups_dev) & set(groups_test)) == 0}")

Total participants: 21
Holding out 3 participants for test set

Dev set:  (16013, 30, 274) -> 18 participants
Test set: (2835, 30, 274) -> 3 participants
Test participants: [27610 49445 61333]

No participant appears in both sets: True


## Group K-Fold Cross Validation

Group k-folds allows us to keep the benefits of a group split participant while stills doing cross validation.
With cross validation we acheive better data usage, more reliable predictions, more generalized hyperparametrs, and an overall mores stable model. Every clip gets to be in validation exactly once rather than permanently reserving a portion of the of data that never trains the model. This is much more stable than a single val split that could be lucky or unlucky. It also reduces overfitting to the validation data.

With N_CV_FOLDS=5 and 18 dev participants:
Each fold uses ~14-15 participants for training and ~3-4 for validation.

We store fold_indices so all three models (RF, CNN, LSTM) use the exact same folds. This makes comparison fair across models.

In [18]:
gkf = GroupKFold(n_splits=N_CV_FOLDS) #creates the group k fold object, defining n_splits as the number of cross validation folds we want

# saves the fold indices for consistent use across all models
fold_indices = []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(X_dev, y_dev, groups=groups_dev)): # loops through each fold generated by the GroupKFold, where tr_idx and va_idx are the indices for the training and validation sets for each respective fold
    
    # store indices for use during model training and hyperparameter tuning
    fold_indices.append((tr_idx, va_idx))
    
    # get the participant IDs in each split for this fold
    tr_participants = set(groups_dev[tr_idx])
    va_participants = set(groups_dev[va_idx])
    
    # verify no participant appears in both train and val for this fold
    assert not (tr_participants & va_participants), f"Fold {fold+1} has participant leakage"
    
    print(f"Fold {fold+1}: "f"train={len(tr_idx):5d} clips / {len(tr_participants):2d} participants  |  "f"val={len(va_idx)} clips / {len(va_participants):2d} participants")

print(f"\n{N_CV_FOLDS} folds created")

Fold 1: train=12503 clips / 14 participants  |  val=3510 clips /  4 participants
Fold 2: train=13185 clips / 15 participants  |  val=2828 clips /  3 participants
Fold 3: train=12638 clips / 14 participants  |  val=3375 clips /  4 participants
Fold 4: train=13189 clips / 15 participants  |  val=2824 clips /  3 participants
Fold 5: train=12537 clips / 14 participants  |  val=3476 clips /  4 participants

5 folds created


## Saving Data


Saving everything needed for training into compressed numpy files. The training notebooks will just load these files.

Arrays saved:
- dev.npz: X (N, 30, 274), y, groups — used for cross validation training
- test.npz: X (N, 30, 274), y, groups — holdout set, never used during training
- cv_folds.npz: fold indices for consistent splits across all three models
- classes.npy: label encoder class names for reversing predictions at inference
- label_encoder.pkl: full label encoder object

In [19]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# save dev set (train + val data) with participant group labels
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'dev.npz'),
    X=X_dev, y=y_dev, groups=groups_dev
)

# save holdout test set, this does not ever get used during training or hyperparameter tuning
np.savez_compressed(
    os.path.join(OUTPUT_DIR, 'test.npz'),
    X=X_test, y=y_test, groups=groups_test
)

# save cross-validation fold indices
np.savez(
    os.path.join(OUTPUT_DIR, 'cv_folds.npz'),
    **{f'fold_{i}_train': tr for i, (tr, _) in enumerate(fold_indices)},
    **{f'fold_{i}_val':   va for i, (_, va) in enumerate(fold_indices)},
)

# save label encoder class names so training/inference notebooks can reverse integer predictions back to sign names without reloading sklearn
np.save(os.path.join(OUTPUT_DIR, 'classes.npy'), le.classes_)

# save full label encoder object 
joblib.dump(le, os.path.join(OUTPUT_DIR, 'label_encoder.pkl'))

# print file sizes to confirm everything saved correctly
print('=== Saved Files ===')
for f in ['dev.npz', 'test.npz', 'cv_folds.npz', 'classes.npy', 'label_encoder.pkl']:
    p = os.path.join(OUTPUT_DIR, f)
    print(f'  {f}: {os.path.getsize(p)/1024/1024:.1f} MB')

=== Saved Files ===
  dev.npz: 308.7 MB
  test.npz: 54.6 MB
  cv_folds.npz: 0.6 MB
  classes.npy: 0.0 MB
  label_encoder.pkl: 0.0 MB
